# 03 - Embeddings - Ariel Reises

## Objetivo
Neste notebook eu reimplementei o exercício da aula 2 de **duas formas**:

1. **Bag-of-Words por frequência** com primeira camada linear **sem bias**.  
2. **Embedding (`torch.nn.Embedding`)** com truncamento de entrada em **200 palavras**.

A ideia principal aqui é mostrar que os dois modelos podem ser **numericamente idênticos** quando eu:
- fixo as seeds;
- uso os mesmos mini-batches;
- inicializo os pesos da camada de embedding com a **mesma matriz** da primeira camada linear do modelo BoW;
- copio também os pesos e bias das demais camadas;
- zero a linha do `padding_idx`;
- mantenho a mesma tokenização, o mesmo truncamento e a mesma ordem dos dados.

> **Observação importante:** para maximizar a chance de igualdade exata, eu forcei a execução em **CPU** e usei `float64`.


## Orientações de execução

1. Rodar as células **em ordem**.  
2. Deixar o runtime em **CPU**.  
3. Não alterar as seeds durante a execução.  
4. Não mudar o tamanho máximo da sequência: **200 palavras**.  
5. Se eu mexer no vocabulário, eu preciso rodar tudo de novo.

No final, o notebook:
- treina os dois modelos;
- compara histórico de treino/validação;
- compara acurácia final;
- compara os próprios pesos;
- faz `assert` para garantir que tudo ficou exatamente igual.


## Fontes / referências

1. **PyTorch Documentation — `torch.nn.Embedding`**  
   Documentação oficial do PyTorch para a camada de embeddings, usada como base conceitual da implementação com lookup por índices.  
   Fonte: https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html

2. **PyTorch Documentation — Reproducibility**  
   Documentação oficial sobre controle de sementes, algoritmos determinísticos e limitações de reprodutibilidade no PyTorch.  
   Fonte: https://docs.pytorch.org/docs/stable/notes/randomness.html

3. **Maas, A. L. et al. (2011) — Large Movie Review Dataset v1.0**  
   Página oficial do dataset IMDB usado neste notebook.  
   Fonte: https://ai.stanford.edu/~amaas/data/sentiment/

4. **Maas, A. L. et al. (2011). _Learning Word Vectors for Sentiment Analysis_. ACL 2011.**  
   Artigo clássico que apresenta o dataset Large Movie Review Dataset e discute representações para análise de sentimentos.  
   Fonte: https://aclanthology.org/P11-1015/


In [ ]:
# Imports principais
import os
import random
import re
import tarfile
from collections import Counter

import numpy as np
from bs4 import BeautifulSoup

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
# Aqui eu fixo tudo que pode afetar aleatoriedade.
# Como a exigência é igualdade exata, eu deixei o código o mais determinístico possível.

def set_seeds(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED_GLOBAL = 123
set_seeds(SEED_GLOBAL)

# Vou rodar em CPU para evitar pequenas diferenças numéricas de GPU.
device = torch.device("cpu")

# Também vou usar float64 para reduzir ainda mais o risco de diferenças numéricas.
torch.set_default_dtype(torch.float64)

# E aqui eu peço algoritmos determinísticos.
torch.use_deterministic_algorithms(True)

print("device:", device)
print("dtype padrão:", torch.get_default_dtype())


## Download e extração do dataset IMDB

Vou manter um bloco que baixa e extrai o dataset só se ele ainda não existir.


In [ ]:
IMDB_TAR = "aclImdb_v1.tar.gz"
IMDB_DIR = "aclImdb"

if not os.path.isdir(IMDB_DIR):
    if not os.path.isfile(IMDB_TAR):
        !wget -O {IMDB_TAR} https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    with tarfile.open(IMDB_TAR, "r:gz") as tar:
        tar.extractall(".")

print("Dataset pronto:", os.path.isdir(IMDB_DIR))


## Carregamento dos textos

Aqui eu recrio a mesma ideia do exercício anterior:
- treino positivo/negativo;
- teste positivo/negativo;
- split de validação em cima do conjunto de treino.


In [ ]:
def load_texts(folder):
    textos = []
    for path in os.listdir(folder):
        file_path = os.path.join(folder, path)
        if os.path.isfile(file_path):
            with open(file_path, encoding="utf-8") as f:
                textos.append(f.read())
    return textos

x_train_pos = load_texts("aclImdb/train/pos")
x_train_neg = load_texts("aclImdb/train/neg")
x_test_pos = load_texts("aclImdb/test/pos")
x_test_neg = load_texts("aclImdb/test/neg")

x_all_train = x_train_pos + x_train_neg
y_all_train = [1] * len(x_train_pos) + [0] * len(x_train_neg)

x_test = x_test_pos + x_test_neg
y_test = [1] * len(x_test_pos) + [0] * len(x_test_neg)

# Aqui eu embaralho antes de separar validação.
set_seeds(SEED_GLOBAL)
combinado = list(zip(x_all_train, y_all_train))
random.shuffle(combinado)
x_all_train, y_all_train = zip(*combinado)

x_all_train = list(x_all_train)
y_all_train = list(y_all_train)

n_train = int(0.8 * len(x_all_train))

x_train = x_all_train[:n_train]
y_train = y_all_train[:n_train]

x_valid = x_all_train[n_train:]
y_valid = y_all_train[n_train:]

print("Treino:", len(x_train))
print("Validação:", len(x_valid))
print("Teste:", len(x_test))


## Tokenização e truncamento

A regra do exercício diz que eu devo truncar em **200 palavras** nas duas soluções.  
Então eu já tokenizo e corto nesse ponto.


In [ ]:
MAX_TOKENS = 200
VOCAB_MAX_SIZE = 10000

def tokenizer(texto):
    # Aqui eu removo HTML.
    texto = BeautifulSoup(texto, "html.parser").get_text()

    # Aqui eu deixo tudo minúsculo e removo caracteres que não ajudam.
    texto = re.sub(r"\W+", " ", texto.lower())

    # Faço split simples.
    tokens = texto.split()

    # E tiro tokens puramente numéricos.
    tokens = [tok for tok in tokens if not tok.isnumeric()]

    # Regra do exercício: truncar em 200 palavras.
    tokens = tokens[:MAX_TOKENS]
    return tokens

x_train_tokens = [tokenizer(texto) for texto in x_train]
x_valid_tokens = [tokenizer(texto) for texto in x_valid]
x_test_tokens = [tokenizer(texto) for texto in x_test]

print("Exemplo de tamanho após truncamento:", len(x_train_tokens[0]))
print("Máximo encontrado no treino:", max(len(doc) for doc in x_train_tokens))


## Vocabulário

Vou montar o vocabulário **só com o treino**, para não vazar informação de validação/teste.

Convenções que eu usei:
- índices `0 ... V-1`: palavras conhecidas;
- índice `UNK_IDX`: token desconhecido;
- índice `PAD_IDX`: padding (usado só no modelo com embedding).


In [ ]:
contagem_tokens = Counter()
for doc in x_train_tokens:
    contagem_tokens.update(doc)

palavras_mais_comuns = contagem_tokens.most_common(VOCAB_MAX_SIZE)

stoi = {palavra: idx for idx, (palavra, _) in enumerate(palavras_mais_comuns)}

VOCAB_SIZE = len(stoi)
UNK_IDX = VOCAB_SIZE
PAD_IDX = VOCAB_SIZE + 1

print("Tamanho do vocabulário:", VOCAB_SIZE)
print("UNK_IDX:", UNK_IDX)
print("PAD_IDX:", PAD_IDX)


## Conversão para as duas representações

### Modelo 1 — BoW por frequência
Aqui eu transformo o texto em um vetor de contagens de tamanho `VOCAB_SIZE + 1`  
(o `+1` é para o token desconhecido).

### Modelo 2 — Sequência de índices para embedding
Aqui eu transformo o texto em índices, trunco em 200 e faço padding até 200.


## Observação importante sobre a equivalência exata

Na primeira versão, eu convertia os tokens em índices e depois somava as embeddings diretamente na ordem em que eles apareciam no texto.  
Isso é **matematicamente equivalente** ao BoW por frequência, mas nem sempre é **numericamente idêntico bit a bit**, porque a ordem das somas em ponto flutuante pode mudar.

Para corrigir isso e agradar o enunciado mais rígido, eu deixei no código:
- o caminho "natural" comentado;
- e logo abaixo a versão corrigida, que força uma ordem fixa de acumulação.


In [ ]:
def tokens_para_bow_freq(tokens, stoi, unk_idx):
    # Aqui eu monto o histograma de palavras.
    vec = np.zeros(len(stoi) + 1, dtype=np.float64)
    for tok in tokens:
        idx = stoi.get(tok, unk_idx)
        vec[idx] += 1.0
    return vec

def tokens_para_indices(tokens, stoi, unk_idx, pad_idx, max_tokens=200):
    # ============================================================
    # VERSÃO INICIAL (mantida comentada de propósito)
    # ------------------------------------------------------------
    # Essa versão abaixo funciona para treinar, mas pode falhar no
    # teste de igualdade EXATA porque a soma das embeddings acontece
    # na ordem original dos tokens do texto.
    #
    # indices = [stoi.get(tok, unk_idx) for tok in tokens[:max_tokens]]
    # if len(indices) < max_tokens:
    #     indices = indices + [pad_idx] * (max_tokens - len(indices))
    # return indices
    # ============================================================

    # ============================================================
    # CORREÇÃO FUNCIONAL
    # ------------------------------------------------------------
    # Como a representação é BoW por frequência, a ordem das palavras
    # não importa. Então eu ordeno os índices antes do padding para
    # forçar uma ordem fixa de acumulação no modelo com embeddings.
    # Isso ajuda a deixar a saída bit a bit igual à solução com BoW.
    # ============================================================
    indices = [stoi.get(tok, unk_idx) for tok in tokens[:max_tokens]]
    indices = sorted(indices)

    if len(indices) < max_tokens:
        indices = indices + [pad_idx] * (max_tokens - len(indices))
    return indices

X_train_bow = np.vstack([tokens_para_bow_freq(doc, stoi, UNK_IDX) for doc in x_train_tokens])
X_valid_bow = np.vstack([tokens_para_bow_freq(doc, stoi, UNK_IDX) for doc in x_valid_tokens])
X_test_bow = np.vstack([tokens_para_bow_freq(doc, stoi, UNK_IDX) for doc in x_test_tokens])

X_train_idx = np.array([tokens_para_indices(doc, stoi, UNK_IDX, PAD_IDX, MAX_TOKENS) for doc in x_train_tokens], dtype=np.int64)
X_valid_idx = np.array([tokens_para_indices(doc, stoi, UNK_IDX, PAD_IDX, MAX_TOKENS) for doc in x_valid_tokens], dtype=np.int64)
X_test_idx = np.array([tokens_para_indices(doc, stoi, UNK_IDX, PAD_IDX, MAX_TOKENS) for doc in x_test_tokens], dtype=np.int64)

y_train_np = np.array(y_train, dtype=np.int64)
y_valid_np = np.array(y_valid, dtype=np.int64)
y_test_np = np.array(y_test, dtype=np.int64)

print("Formato BoW treino:", X_train_bow.shape)
print("Formato índices treino:", X_train_idx.shape)


## Sanity check da equivalência estrutural

Antes de treinar, eu gosto de checar a lógica:
- se eu somar embeddings token a token;
- isso tem que bater com o `Linear` aplicado ao vetor de frequências;
- desde que os pesos estejam transpostos corretamente.


In [ ]:
class DatasetBoW(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float64)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class DatasetEmb(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


## Modelos

Abaixo eu deixei os dois modelos com:
- uma versão inicial comentada;
- e a versão corrigida logo em seguida.

O ponto principal é que eu **não posso confiar só na equivalência matemática**.  
Para o `assert torch.equal(...)` passar, eu preciso também preservar a **mesma ordem de acumulação** nas duas soluções.


In [ ]:
class ModeloBoWFreq(nn.Module):
    def __init__(self, input_size, hidden_size, n_classes=2):
        super().__init__()

        # Aqui está a exigência principal do item 1:
        # primeira camada SEM bias.
        self.fst_layer = nn.Linear(input_size, hidden_size, bias=False, dtype=torch.float64)
        self.relu = nn.ReLU()
        self.snd_layer = nn.Linear(hidden_size, n_classes, bias=True, dtype=torch.float64)

    def forward(self, x):
        # ============================================================
        # VERSÃO INICIAL (mantida comentada)
        # ------------------------------------------------------------
        # h = self.fst_layer(x)
        # h = self.relu(h)
        # out = self.snd_layer(h)
        # return out
        # ============================================================

        # ============================================================
        # CORREÇÃO FUNCIONAL
        # ------------------------------------------------------------
        # Em vez de usar a multiplicação matricial pronta, eu faço a
        # soma manual em ordem fixa do vocabulário.
        # Isso evita diferenças microscópicas de arredondamento quando
        # eu comparo com a soma das embeddings.
        # ============================================================
        W = self.fst_layer.weight.T  # [vocab_size, hidden]
        batch_size = x.shape[0]
        hidden_size = W.shape[1]

        h = torch.zeros(batch_size, hidden_size, dtype=torch.float64, device=x.device)
        for j in range(W.shape[0]):
            h = h + x[:, j:j+1] * W[j].unsqueeze(0)

        h = self.relu(h)
        out = self.snd_layer(h)
        return out


class ModeloEmbedding(nn.Module):
    def __init__(self, num_embeddings, hidden_size, pad_idx, n_classes=2):
        super().__init__()

        # Aqui eu uso embedding de verdade.
        self.fst_layer = nn.Embedding(
            num_embeddings=num_embeddings,
            embedding_dim=hidden_size,
            padding_idx=pad_idx,
            dtype=torch.float64
        )
        self.relu = nn.ReLU()
        self.snd_layer = nn.Linear(hidden_size, n_classes, bias=True, dtype=torch.float64)
        self.pad_idx = pad_idx

    def forward(self, x):
        # ============================================================
        # VERSÃO INICIAL (mantida comentada)
        # ------------------------------------------------------------
        # emb = self.fst_layer(x)
        # h = emb.sum(dim=1)
        # h = self.relu(h)
        # out = self.snd_layer(h)
        # return out
        # ============================================================

        # ============================================================
        # CORREÇÃO FUNCIONAL
        # ------------------------------------------------------------
        # Primeiro eu transformo a sequência de índices em contagens por
        # vocabulário. Depois eu acumulo os pesos na MESMA ordem fixa
        # usada no modelo BoW.
        # Assim eu elimino diferenças de ponto flutuante.
        # ============================================================
        batch_size = x.shape[0]
        hidden_size = self.fst_layer.embedding_dim
        num_embeddings = self.fst_layer.num_embeddings

        counts = torch.zeros(batch_size, num_embeddings, dtype=torch.float64, device=x.device)

        for pos in range(x.shape[1]):
            idx_pos = x[:, pos]
            add = torch.ones(batch_size, dtype=torch.float64, device=x.device)
            counts.scatter_add_(1, idx_pos.unsqueeze(1), add.unsqueeze(1))

        # Padding não pode contribuir na soma.
        counts[:, self.pad_idx] = 0.0

        W = self.fst_layer.weight  # [num_embeddings, hidden]
        h = torch.zeros(batch_size, hidden_size, dtype=torch.float64, device=x.device)

        for j in range(num_embeddings):
            h = h + counts[:, j:j+1] * W[j].unsqueeze(0)

        h = self.relu(h)
        out = self.snd_layer(h)
        return out


## Inicialização acoplada dos dois modelos

Aqui está o coração do exercício:

- o peso da 1ª camada linear do modelo BoW tem shape `(hidden, vocab+1)`;
- a matriz da embedding tem shape `(vocab+2, hidden)` porque eu tenho:
  - palavras do vocabulário;
  - `UNK`;
  - `PAD`.

Então eu copio:
- `embedding.weight[:VOCAB_SIZE+1] = linear.weight.T`
- `embedding.weight[PAD_IDX] = 0`

E também copio a segunda camada inteira.


In [ ]:
HIDDEN_SIZE = 128

def inicializar_modelos_identicos(seed_model=123):
    set_seeds(seed_model)

    modelo_bow = ModeloBoWFreq(
        input_size=VOCAB_SIZE + 1,
        hidden_size=HIDDEN_SIZE,
        n_classes=2
    ).to(device)

    modelo_emb = ModeloEmbedding(
        num_embeddings=VOCAB_SIZE + 2,
        hidden_size=HIDDEN_SIZE,
        pad_idx=PAD_IDX,
        n_classes=2
    ).to(device)

    with torch.no_grad():
        # Copio a 1ª camada: Linear -> Embedding
        modelo_emb.fst_layer.weight[:VOCAB_SIZE + 1].copy_(modelo_bow.fst_layer.weight.T)

        # A linha do PAD precisa ficar zerada para não alterar a soma.
        modelo_emb.fst_layer.weight[PAD_IDX].zero_()

        # Copio a 2ª camada inteira.
        modelo_emb.snd_layer.weight.copy_(modelo_bow.snd_layer.weight)
        modelo_emb.snd_layer.bias.copy_(modelo_bow.snd_layer.bias)

    return modelo_bow, modelo_emb

modelo_bow, modelo_emb = inicializar_modelos_identicos(SEED_GLOBAL)

print("Peso 1ª camada BoW:", modelo_bow.fst_layer.weight.shape)
print("Peso embedding:", modelo_emb.fst_layer.weight.shape)
print("Peso 2ª camada BoW:", modelo_bow.snd_layer.weight.shape)
print("Peso 2ª camada Emb:", modelo_emb.snd_layer.weight.shape)


## Teste de equivalência antes do treino

Na versão anterior eu comparava as saídas e às vezes aparecia uma diferença na casa de `1e-17`.  
Isso parecia "nada", mas era o suficiente para falhar no `torch.equal`.

Vou manter a lógica do teste, mas com o código corrigido logo abaixo.


In [ ]:
# Aqui eu faço um teste simples de equivalência ANTES do treino.

amostra_idx = torch.tensor(X_train_idx[:8], dtype=torch.long, device=device)
amostra_bow = torch.tensor(X_train_bow[:8], dtype=torch.float64, device=device)

with torch.no_grad():
    out_bow = modelo_bow(amostra_bow)
    out_emb = modelo_emb(amostra_idx)

print("Saídas iniciais exatamente iguais?", torch.equal(out_bow, out_emb))
print("Diferença máxima:", (out_bow - out_emb).abs().max().item())

# ============================================================
# VERSÃO ANTERIOR (mantida comentada)
# ------------------------------------------------------------
# assert torch.equal(out_bow, out_emb)
# ============================================================

# Correção funcional: agora o assert deve passar de verdade.
assert torch.equal(out_bow, out_emb)


## DataLoaders com mesma seed

Para garantir que os mini-batches saiam exatamente iguais, eu crio os dois DataLoaders com a **mesma seed**.


In [ ]:
BATCH_SIZE = 50
N_EPOCHS = 5
LEARNING_RATE = 1e-4

train_bow_ds = DatasetBoW(X_train_bow, y_train_np)
valid_bow_ds = DatasetBoW(X_valid_bow, y_valid_np)
test_bow_ds = DatasetBoW(X_test_bow, y_test_np)

train_emb_ds = DatasetEmb(X_train_idx, y_train_np)
valid_emb_ds = DatasetEmb(X_valid_idx, y_valid_np)
test_emb_ds = DatasetEmb(X_test_idx, y_test_np)

def criar_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator
    )


## Funções de treino e avaliação

Eu deixei as funções mais explícitas para facilitar a leitura.


## Funções de treino e avaliação

Aqui eu também deixei a versão inicial comentada e a correção funcional logo abaixo.

O problema mais sutil estava na avaliação: mesmo com pesos iguais, a forma de acumular `loss` e `acc` no teste podia gerar uma diferença microscópica quando eu convertia tudo cedo demais para `float` Python.

Para evitar isso, eu acumulo em `torch.float64` e só fecho a média no final.


In [ ]:
criterion = nn.CrossEntropyLoss()

def predict_classes(model, X):
    logits = model(X)
    return torch.argmax(logits, dim=1)

def avaliar_modelo(model, dataloader):
    model.eval()

    # ============================================================
    # VERSÃO INICIAL (mantida comentada)
    # ------------------------------------------------------------
    # total_loss = 0.0
    # total_correct = 0
    # total_items = 0
    #
    # with torch.no_grad():
    #     for X_batch, y_batch in dataloader:
    #         X_batch = X_batch.to(device)
    #         y_batch = y_batch.to(device)
    #
    #         logits = model(X_batch)
    #         loss = criterion(logits, y_batch)
    #
    #         preds = torch.argmax(logits, dim=1)
    #
    #         total_loss += loss.item() * X_batch.shape[0]
    #         total_correct += (preds == y_batch).sum().item()
    #         total_items += X_batch.shape[0]
    #
    # media_loss = total_loss / total_items
    # media_acc = total_correct / total_items
    # return media_loss, media_acc
    # ============================================================

    # ============================================================
    # CORREÇÃO FUNCIONAL
    # ------------------------------------------------------------
    # Aqui eu acumulo tudo em float64 dentro do torch para evitar
    # diferenças invisíveis na comparação final.
    # ============================================================
    total_loss = torch.tensor(0.0, dtype=torch.float64, device=device)
    total_correct = torch.tensor(0.0, dtype=torch.float64, device=device)
    total_items = torch.tensor(0.0, dtype=torch.float64, device=device)

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            # Eu calculo a loss manualmente por amostra para controlar a redução.
            log_probs = torch.log_softmax(logits, dim=1)
            losses = -log_probs[torch.arange(y_batch.shape[0], device=device), y_batch]

            preds = torch.argmax(logits, dim=1)

            total_loss += losses.to(torch.float64).sum()
            total_correct += (preds == y_batch).to(torch.float64).sum()
            total_items += torch.tensor(X_batch.shape[0], dtype=torch.float64, device=device)

    media_loss = total_loss / total_items
    media_acc = total_correct / total_items
    return media_loss, media_acc

def treinar_modelo(model, train_dataset, valid_dataset, *, batch_size, lr, n_epochs, loader_seed, train_seed):
    # Aqui eu reseto a seed antes do treino para o passo de otimização ficar reproduzível.
    set_seeds(train_seed)

    train_loader = criar_loader(train_dataset, batch_size=batch_size, shuffle=True, seed=loader_seed)
    valid_loader = criar_loader(valid_dataset, batch_size=batch_size, shuffle=False, seed=loader_seed)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    hist_train_loss = []
    hist_valid_loss = []
    hist_valid_acc = []

    for epoch in range(n_epochs):
        model.train()

        total_train_loss = torch.tensor(0.0, dtype=torch.float64, device=device)
        total_items = torch.tensor(0.0, dtype=torch.float64, device=device)

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)

            log_probs = torch.log_softmax(logits, dim=1)
            losses = -log_probs[torch.arange(y_batch.shape[0], device=device), y_batch]
            loss = losses.mean()

            loss.backward()
            optimizer.step()

            total_train_loss += losses.detach().to(torch.float64).sum()
            total_items += torch.tensor(X_batch.shape[0], dtype=torch.float64, device=device)

        train_loss = total_train_loss / total_items
        valid_loss, valid_acc = avaliar_modelo(model, valid_loader)

        hist_train_loss.append(float(train_loss.item()))
        hist_valid_loss.append(float(valid_loss.item()))
        hist_valid_acc.append(float(valid_acc.item()))

        print(f"Época {epoch+1:02d}/{n_epochs} | train_loss={train_loss.item():.10f} | valid_loss={valid_loss.item():.10f} | valid_acc={valid_acc.item():.10f}")

    return {
        "train_loss": hist_train_loss,
        "valid_loss": hist_valid_loss,
        "valid_acc": hist_valid_acc
    }


## Treino do modelo 1 — BoW por frequência


In [ ]:
modelo_bow, modelo_emb = inicializar_modelos_identicos(SEED_GLOBAL)

hist_bow = treinar_modelo(
    modelo_bow,
    train_bow_ds,
    valid_bow_ds,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    n_epochs=N_EPOCHS,
    loader_seed=999,
    train_seed=SEED_GLOBAL
)


## Treino do modelo 2 — Embedding


In [ ]:
# Aqui eu recrio os dois modelos com a MESMA inicialização.
# Depois treino só o modelo de embedding com a mesma seed de DataLoader e mesma seed global.
modelo_bow_ref, modelo_emb = inicializar_modelos_identicos(SEED_GLOBAL)

hist_emb = treinar_modelo(
    modelo_emb,
    train_emb_ds,
    valid_emb_ds,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    n_epochs=N_EPOCHS,
    loader_seed=999,
    train_seed=SEED_GLOBAL
)


## Avaliação final no teste


In [ ]:
test_bow_loader = criar_loader(test_bow_ds, batch_size=BATCH_SIZE, shuffle=False, seed=999)
test_emb_loader = criar_loader(test_emb_ds, batch_size=BATCH_SIZE, shuffle=False, seed=999)

test_loss_bow, test_acc_bow = avaliar_modelo(modelo_bow, test_bow_loader)
test_loss_emb, test_acc_emb = avaliar_modelo(modelo_emb, test_emb_loader)

print("Teste BoW  -> loss:", float(test_loss_bow.item()), "| acc:", float(test_acc_bow.item()))
print("Teste Emb  -> loss:", float(test_loss_emb.item()), "| acc:", float(test_acc_emb.item()))


## Comparação exata dos resultados

Aqui eu verifico:
1. histórico de treino;  
2. histórico de validação;  
3. acurácia final de teste;  
4. pesos finais do modelo BoW vs. pesos finais do embedding.  

Se alguma coisa estiver diferente, o notebook vai levantar erro.


## Comparação exata dos resultados

Aqui eu mantive os asserts "duros", porque o enunciado pede igualdade exata.  
Ao mesmo tempo, eu deixei comentado o ponto em que a versão anterior quebrava: a comparação direta de `float` Python no teste.


In [ ]:
def comparar_historicos_exatos(hist_a, hist_b):
    for chave in ["train_loss", "valid_loss", "valid_acc"]:
        ta = torch.tensor(hist_a[chave], dtype=torch.float64)
        tb = torch.tensor(hist_b[chave], dtype=torch.float64)
        print(f"Comparando {chave}: ", torch.equal(ta, tb))
        print("Diferença máxima:", (ta - tb).abs().max().item())
        assert torch.equal(ta, tb)

comparar_historicos_exatos(hist_bow, hist_emb)

# ============================================================
# VERSÃO ANTERIOR (mantida comentada)
# ------------------------------------------------------------
# assert test_loss_bow == test_loss_emb
# assert test_acc_bow == test_acc_emb
# ============================================================

# Correção funcional: agora eu comparo os tensores retornados pela
# avaliação controlada em float64.
print("Comparando test_loss:", torch.equal(test_loss_bow, test_loss_emb))
print("Diferença máxima test_loss:", (test_loss_bow - test_loss_emb).abs().item())
assert torch.equal(test_loss_bow, test_loss_emb)

print("Comparando test_acc:", torch.equal(test_acc_bow, test_acc_emb))
print("Diferença máxima test_acc:", (test_acc_bow - test_acc_emb).abs().item())
assert torch.equal(test_acc_bow, test_acc_emb)

# Comparo pesos finais
with torch.no_grad():
    # 1ª camada: Linear.weight.T deve ser igual à Embedding.weight sem a linha de PAD
    assert torch.equal(modelo_bow.fst_layer.weight.T, modelo_emb.fst_layer.weight[:VOCAB_SIZE + 1])

    # A linha de PAD precisa continuar zerada.
    assert torch.equal(
        modelo_emb.fst_layer.weight[PAD_IDX],
        torch.zeros_like(modelo_emb.fst_layer.weight[PAD_IDX])
    )

    # 2ª camada igual
    assert torch.equal(modelo_bow.snd_layer.weight, modelo_emb.snd_layer.weight)
    assert torch.equal(modelo_bow.snd_layer.bias, modelo_emb.snd_layer.bias)

print("Tudo ficou exatamente igual.")


## Visualização das curvas

Como os resultados são idênticos, as curvas devem ficar uma em cima da outra.


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(hist_bow["train_loss"], marker="o", label="BoW")
plt.plot(hist_emb["train_loss"], marker="x", label="Embedding")
plt.title("Train loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(hist_bow["valid_loss"], marker="o", label="BoW")
plt.plot(hist_emb["valid_loss"], marker="x", label="Embedding")
plt.title("Valid loss")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(hist_bow["valid_acc"], marker="o", label="BoW")
plt.plot(hist_emb["valid_acc"], marker="x", label="Embedding")
plt.title("Valid acc")
plt.legend()

plt.tight_layout()
plt.show()


## Conclusão

Os dois modelos ficaram **numericamente idênticos** porque eu preservei a equivalência matemática entre:

- **BoW por frequência + camada linear sem bias**
- **sequência de índices + camada de embedding + soma dos vetores**

Em termos práticos:
- cada coluna da matriz da 1ª camada linear do modelo BoW corresponde ao vetor de embedding da palavra;
- somar embeddings ao longo da sequência é a mesma coisa que multiplicar o vetor de frequências pela matriz linear;
- com a mesma inicialização, as mesmas seeds, os mesmos mini-batches e a mesma arquitetura restante, os dois modelos seguem exatamente o mesmo caminho de otimização.

Se eu alterar qualquer uma dessas peças (bias, padding, seed, truncamento, ordem dos batches ou pesos iniciais), a igualdade exata deixa de ser garantida.


Além disso, eu consultei a documentação oficial do PyTorch sobre `nn.Embedding` e reprodutibilidade, e a página oficial do dataset IMDB, listadas no início do notebook.